# Face anti-spoofing for eKYC fraud: before vs baseline vs ours

| Run | What it is |
|---|---|
| `before` | Minivision's pretrained MiniFASNet used as-is |
| `baseline` | fine-tuned with the original recipe (standard augmentation + FFT head + cross-entropy) |
| `ours` | fine-tuned with fraud-aware augmentation + FFT head + live-vs-attack loss |
| `ours_no_*`, `ours_ce_loss` | ablations: `ours` with one part removed |

**Before running:** Accelerator = GPU, Internet = On, Add Input = your `fas-project` dataset (`python scripts/pack_kaggle.py`).
CelebA-Spoof is for non-commercial research only; do not publish its images.

In [ ]:
import os
TRAIN_FILES, VALID_FILES, TEST_FILES = 10, 1, 6   # ~45k train, ~5k val, ~5k test faces
EPOCHS, BATCH, WORKERS = 12, 256, os.cpu_count()   # Kaggle GPU machines have 4 CPUs
RUN_ABLATIONS = True
DATA = "/tmp/fas_data"   # outside /kaggle/working so crops are not saved with the notebook

In [ ]:
import glob, os, shutil
src = next(os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/fas/__init__.py", recursive=True))
shutil.copytree(src, "/kaggle/working/fas-project", dirs_exist_ok=True)
os.chdir("/kaggle/working/fas-project")
!pip install -q onnx onnxruntime onnxscript
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn GPU on")

## 1. Data

In [ ]:
!python scripts/prepare_data.py --train {TRAIN_FILES} --valid {VALID_FILES} --test {TEST_FILES} --raw /tmp/celeba_raw --out {DATA}

## 2. Train and evaluate

In [ ]:
RUNS = {"baseline": "--aug standard --loss ce", "ours": "--aug fraud --loss live_vs_attack"}
ABLATIONS = {
    "ours_no_fft":       "--aug fraud --loss live_vs_attack --no_fft",
    "ours_no_synthetic": "--aug fraud --loss live_vs_attack --disable moire screen_grid glare halftone_print",
    "ours_no_degrade":   "--aug fraud --loss live_vs_attack --disable jpeg noise low_light blur low_res white_balance",
    "ours_ce_loss":      "--aug fraud --loss ce",
}
COMMON = f"--epochs {EPOCHS} --batch_size {BATCH} --workers {WORKERS} --train_root {DATA}/train --val_root {DATA}/val"
EVAL = f"--test {DATA}/test --val {DATA}/val"
!python scripts/evaluate.py --models pretrained/anti_spoof --tag before {EVAL}
for tag, flags in {**RUNS, **(ABLATIONS if RUN_ABLATIONS else {})}.items():
    !python scripts/train.py {flags} {COMMON} --out results/runs/{tag}
    !python scripts/evaluate.py --models results/runs/{tag}/models --tag {tag} {EVAL}

## 3. Deployment: ONNX float32 + INT8 (speed, size, accuracy)

In [ ]:
!python scripts/export.py --models pretrained/anti_spoof --name original --out results/app_models --calib {DATA}/val --test {DATA}/test
!python scripts/export.py --models results/runs/ours/models --name ours --out results/app_models --calib {DATA}/val --test {DATA}/test --metrics results/eval/ours/metrics.json

## 4. Results, impact per 1M checks, figures

In [ ]:
!python scripts/report.py --order before baseline ours ours_no_fft ours_no_synthetic ours_no_degrade ours_ce_loss --train {DATA}/train --test {DATA}/test
from IPython.display import Image, Markdown, display
display(Markdown(open("results/summary.md").read()))
for f in sorted(glob.glob("results/figures/*.png")):
    display(Markdown(f"**{os.path.basename(f)}**"), Image(f))
shutil.make_archive("/kaggle/working/fas_results", "zip", "results")
print("Download fas_results.zip from Output; unzip into the project as results/; README step 3 says where the app models go")